# Agentic AI for E-commerce Sales and Customer Insights

## 1. Project Overview

This project uses the Brazilian E-commerce Public Dataset (Olist) to demonstrate how data analytics and AI can support business decision-making in a real marketplace context.

The approach follows a structured progression: from data preparation and exploratory analysis to the development of a rule-based insight engine that simulates an AI-powered business analyst.

The project follows a structured approach, starting with traditional business analytics and evolving into an AI-assisted insight generation system.

The analysis focuses on key business drivers, including sales performance, customer satisfaction, delivery efficiency and marketplace dynamics.

The final objective is to build a Business Analytics Agent capable of:
- answering business questions
- identifying key performance drivers
- generating actionable, data-driven recommendations

The final objective is to build a Business Analytics Agent capable of answering business questions, identifying key drivers and generating actionable recommendations based on real marketplace data.





### 1.1. Why this project matters

E-commerce and marketplace companies rely heavily on data to make decisions across revenue growth, logistics optimization and customer experience.

For Data Analysts and Business Analysts, the value lies not only in building dashboards or models, but in translating data into clear and actionable business insights.

This project demonstrates the ability to:

- perform data cleaning and preparation on real-world datasets
- conduct exploratory and business-focused analysis
- design metrics aligned with business performance
- generate structured insights and recommendations
- simulate an AI-assisted business analyst workflow
- communicate findings in a clear and professional format

This approach reflects how modern data teams operate, where the goal is not only analysis, but enabling better business decisions.

## 2. Import Core Libraries

In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 3. Data Loading

At this stage, the objective is to load each relational table and inspect its dimensions before joining them into an analytical dataset.The dataset is composed of multiple relational tables, similar to what an analyst would encounter in a real business environment.

Instead of working with a single flat file, this project requires joining tables related to orders, customers, products, payments, sellers and reviews. This structure mirrors real-world business environments, where analysts must combine multiple data sources to answer operational and strategic questions.

The dataset files were stored locally in the project folder under `data/raw/`. This structure makes the project easier to reproduce and publish on other platforms.

### 3.1. Import Google Drive

In [97]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 3.2. Set Base Path

In [98]:
base_path = "/content/drive/MyDrive/ai-ecommerce-insights-agent/data/raw/"

files = os.listdir(base_path)
files

['olist_sellers_dataset.csv',
 'product_category_name_translation.csv',
 'olist_products_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_customers_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_payments_dataset.csv']

### 3.3. Load Data

In [99]:
orders = pd.read_csv(base_path + "olist_orders_dataset.csv")
order_items = pd.read_csv(base_path + "olist_order_items_dataset.csv")
customers = pd.read_csv(base_path + "olist_customers_dataset.csv")
products = pd.read_csv(base_path + "olist_products_dataset.csv")
reviews = pd.read_csv(base_path + "olist_order_reviews_dataset.csv")
payments = pd.read_csv(base_path + "olist_order_payments_dataset.csv")
sellers = pd.read_csv(base_path + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(base_path + "product_category_name_translation.csv")

### 3.4. Check Data Loading

In [100]:
datasets = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "products": products,
    "reviews": reviews,
    "payments": payments,
    "sellers": sellers,
    "category_translation": category_translation
}

for name, df in datasets.items():
    print(name, df.shape)

orders (99441, 8)
order_items (112650, 7)
customers (99441, 5)
products (32951, 9)
reviews (99224, 7)
payments (103886, 5)
sellers (3095, 4)
category_translation (71, 2)


## 4. Building the Analytical Dataset

The dataset is composed of multiple relational tables. In order to perform business analysis, it is necessary to combine these tables into a single analytical dataset.

This process simulates real-world data work, where analysts often need to integrate data from different sources before generating insights.

The central table is the orders dataset, which is enriched with:
- item-level information (products and prices)
- customer data (location)
- payment data
- review scores
- product categories

The result is a unified dataset that allows analysis of sales performance, customer satisfaction and operational efficiency.

A key challenge in the dataset is that orders can contain multiple items and payments, which can lead to duplication when calculating metrics.

To ensure accurate analysis, an order-level dataset is created, preventing inflated counts and biased averages.

### 4.1. Orders + Customers

In [101]:
df = orders.merge(customers, on="customer_id", how="left")

### 4.2. Add Order Items

In [102]:
df = df.merge(order_items, on="order_id", how="left")

### 4.3. Add Payments

In [103]:
df = df.merge(payments, on="order_id", how="left")

### 4.4. Add Reviews

In [104]:
df = df.merge(reviews[["order_id", "review_score"]], on="order_id", how="left")

### 4.5. Add Products

In [105]:
df = df.merge(products, on="product_id", how="left")

### 4.6. Add Category Translation

In [106]:
df = df.merge(category_translation, on="product_category_name", how="left")

### 4.7. Check Results

In [107]:
df.shape
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value,review_score,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,1.0,credit_card,1.0,18.12,4.0,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,3.0,voucher,1.0,2.00,4.0,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,2.0,voucher,1.0,18.59,4.0,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0,housewares
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,1.0,boleto,1.0,141.46,4.0,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0,perfumery
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,1.0,credit_card,3.0,179.12,5.0,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0,auto


### 4.8. Data Validation and Granularity Adjustment

The initial merged dataset is at the order-item level. This is useful for product and category analysis, but not ideal for order-level metrics such as delivery time, delivery delay and review score.

Because a single order may contain multiple items and multiple payment records, calculating average delivery performance directly from the merged table can duplicate orders and distort results.

To avoid misleading conclusions, this project creates an order-level analytical dataset for delivery and satisfaction analysis.

In [108]:
# Keep only delivered orders with a valid delivery date
df_valid = df[
    (df["order_status"] == "delivered") &
    (df["order_delivered_customer_date"].notna())
].copy()

# Create order-level dataset to avoid duplicated orders
order_level = (
    df_valid
    .groupby("order_id")
    .agg({
        "order_purchase_timestamp": "first",
        "order_delivered_customer_date": "first",
        "order_estimated_delivery_date": "first",
        "review_score": "first",
        "customer_state": "first"
    })
    .reset_index()
)

order_level.shape

(96470, 6)

This separation ensures that operational metrics are calculated at the correct level of granularity, preventing bias introduced by duplicated records.

## 5. Data Processing

### 5.1. Date Processing

To analyze delivery performance and operational efficiency, it is necessary to convert date columns into proper datetime format.

This allows the calculation of metrics such as delivery time and delays, which are critical for evaluating logistics performance in e-commerce.

In [109]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

### 5.2. Business Metrics Design

To enable meaningful business analysis, raw transactional data is transformed into standardized metrics that capture operational performance and customer experience.

In order to generate meaningful insights, raw data must be transformed into business metrics.

The following metrics are created:
- delivery_time: actual time between purchase and delivery
- estimated_delivery_time: expected delivery time
- delivery_delay: difference between actual and estimated delivery
- revenue: total value per item including freight

In [110]:
date_cols_order_level = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols_order_level:
    order_level[col] = pd.to_datetime(order_level[col], errors='coerce')

In [111]:
# Delivery time in days
order_level["delivery_time"] = (
    order_level["order_delivered_customer_date"] - order_level["order_purchase_timestamp"]
).dt.days

# Estimated delivery time in days
order_level["estimated_delivery_time"] = (
    order_level["order_estimated_delivery_date"] - order_level["order_purchase_timestamp"]
).dt.days

# Delivery delay in days
order_level["delivery_delay"] = (
    order_level["order_delivered_customer_date"] - order_level["order_estimated_delivery_date"]
).dt.days

# Revenue remains at item level because it depends on product and freight values
df["revenue"] = df["price"] + df["freight_value"]

## 6. Performance Analysis

### 6.1. Sales Performance Overview

Revenue is one of the most important metrics in any business. The first step is to understand overall revenue performance and distribution across categories.

In [112]:
total_revenue = df["revenue"].sum()
avg_order_value = df.groupby("order_id")["revenue"].sum().mean()

print("Total Revenue:", round(total_revenue, 2))
print("Average Order Value:", round(avg_order_value, 2))

Total Revenue: 16643731.3
Average Order Value: 167.37


This indicates the overall scale of marketplace activity and provides a baseline for evaluating category-level and operational performance.

### 6.2. Revenue by Product Category

Understanding which categories drive revenue helps identify core business segments and opportunities for growth.

In [113]:
revenue_by_category = (
    df.groupby("product_category_name_english")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

revenue_by_category

,revenue
product_category_name_english,
health_beauty,1491397.76
watches_gifts,1358845.59
bed_bath_table,1327662.02
sports_leisure,1205197.85
computers_accessories,1104362.03
furniture_decor,955367.22
housewares,823623.50
cool_stuff,752702.21
auto,714431.95


Revenue is concentrated in a limited number of categories, suggesting both strong core segments and potential dependency risk.

### 6.3. Delivery Performance

Delivery speed and reliability are key drivers of customer satisfaction in e-commerce. Delays may negatively impact customer reviews and retention.

In [114]:
avg_delivery_time = order_level["delivery_time"].mean()
avg_delay = order_level["delivery_delay"].mean()

print("Average Delivery Time:", round(avg_delivery_time, 2))
print("Average Delay:", round(avg_delay, 2))

Average Delivery Time: 12.09
Average Delay: -11.88


A negative delay indicates that orders are delivered earlier than the estimated delivery date, suggesting conservative delivery estimates or strong logistics performance.

### 6.4. Impact of Delivery Delays on Customer Satisfaction

This analysis evaluates whether delivery delays are associated with lower customer review scores, which can indicate operational issues affecting customer experience.

In [115]:
delay_vs_review = order_level.groupby("review_score")["delivery_delay"].mean()

delay_vs_review

,delivery_delay
review_score,
1.0,-4.026544
2.0,-8.622990
3.0,-10.779084
4.0,-12.383264
5.0,-13.382226


There is a clear trend where higher review scores are associated with earlier deliveries. Orders with 5-star reviews are delivered significantly earlier than those with lower ratings.

### 6.5. Key insights

1. Strong delivery performance:
Orders are delivered on average approximately 12 days earlier than the estimated delivery date, indicating a conservative estimation strategy and efficient logistics operations. This suggests that delivery estimates may be systematically overestimated, potentially reducing perceived delivery reliability signals.

2. Delivery timing and customer satisfaction:
There is a clear relationship between delivery performance and review scores. Orders with higher review scores tend to be delivered significantly earlier than expected.

3. Expectation management:
Customer satisfaction is influenced not only by avoiding delays, but by exceeding delivery expectations.

### 6.6. Business implications

- Delivery performance should be treated as a strategic lever for customer experience
- Managing customer expectations (estimated delivery dates) is as important as operational speed
- Monitoring delivery performance relative to expectations provides more actionable insight than absolute delivery time

## 7. Rule-Based Insight Engine

After validating the core business metrics, this section implements a rule based insight generation engine.

The objective is to simulate the logic of an AI assisted business analyst before introducing any paid API or external LLM. This makes the system transparent, auditable and cost free.

Instead of only calculating metrics, the engine interprets results and returns business oriented insights. This creates the foundation for a future agentic AI workflow, where the system can analyze data, identify relevant patterns and recommend actions.

This approach ensures that insights are consistent, scalable and aligned with business interpretation, rather than relying on manual analysis.

### 7.1. Define Insight Generation Functions

Instead of manually interpreting metrics, this section implements a rule-based system that converts numerical outputs into structured business insights.

Each function encapsulates analytical logic, translating data patterns into:
- insight
- business implication
- recommended action

This design simulates how a business analyst interprets data in practice.

In [116]:
def delivery_performance_insight(order_level):
    avg_delay = order_level["delivery_delay"].mean()
    avg_delivery_time = order_level["delivery_time"].mean()

    if avg_delay < -7:
        return {
            "area": "Delivery Performance",
            "insight": (
                f"Delivery performance relative to estimated dates is strong. Orders are delivered on average "
                f"{abs(avg_delay):.1f} days earlier than estimated, with an average "
                f"delivery time of {avg_delivery_time:.1f} days."
            ),
            "business_implication": (
                "The company appears to perform well against estimated delivery dates, possibly due to "
                "conservative estimates, efficient logistics, or both."
            ),
            "recommended_action": (
                "Continue monitoring delivery performance by region and seller to identify where "
                "early delivery contributes most to customer satisfaction."
            )
        }

    elif avg_delay < 0:
        return {
            "area": "Delivery Performance",
            "insight": (
                f"Deliveries are generally ahead of schedule, arriving on average "
                f"{abs(avg_delay):.1f} days before the estimated date."
            ),
            "business_implication": (
                "Delivery performance relative to estimated dates is positive, but the margin against expectations is moderate."
            ),
            "recommended_action": (
                "Investigate whether specific regions or sellers are underperforming relative to the average."
            )
        }

    else:
        return {
            "area": "Delivery Performance",
            "insight": (
                f"Deliveries are delayed on average by {avg_delay:.1f} days."
            ),
            "business_implication": (
                "Late deliveries may negatively affect customer satisfaction and repeat purchases."
            ),
            "recommended_action": (
                "Prioritize logistics diagnostics by region, seller and product category."
            )
        }


def satisfaction_insight(order_level):
    delay_by_review = order_level.groupby("review_score")["delivery_delay"].mean()

    one_star_delay = delay_by_review.loc[1]
    five_star_delay = delay_by_review.loc[5]
    difference = abs(five_star_delay - one_star_delay)

    if five_star_delay < one_star_delay:
        return {
            "area": "Customer Satisfaction",
            "insight": (
                f"Five star orders are delivered {difference:.1f} days earlier relative "
                f"to expectations than one star orders."
            ),
            "business_implication": (
                "Customer satisfaction is associated with delivery performance relative to expectations, "
                "although other factors such as product quality, seller reliability and freight cost may also influence reviews."
            ),
            "recommended_action": (
                "Use delivery performance relative to estimated dates as a customer experience KPI and expand the analysis "
                "to include product category, seller performance and freight value."
            )
        }

    else:
        return {
            "area": "Customer Satisfaction",
            "insight": (
                "Delivery timing does not show a clear positive relationship with review scores."
            ),
            "business_implication": (
                "Other factors such as product quality, seller reliability or pricing may be stronger drivers of reviews."
            ),
            "recommended_action": (
                "Expand the satisfaction analysis to include product category, seller performance and freight value."
            )
        }


def revenue_category_insight(df):
    revenue_by_category = (
        df.groupby("product_category_name_english")["revenue"]
        .sum()
        .sort_values(ascending=False)
    )

    top_category = revenue_by_category.index[0]
    top_revenue = revenue_by_category.iloc[0]
    total_revenue = revenue_by_category.sum()
    top_share = (top_revenue / total_revenue) * 100

    return {
        "area": "Revenue Concentration",
        "insight": (
            f"The top revenue generating category is '{top_category}', generating "
            f"{top_revenue:,.2f} and representing {top_share:.1f}% of analyzed revenue."
        ),
        "business_implication": (
            "Revenue is concentrated in key categories, which may represent both growth opportunities "
            "and dependency risks."
        ),
        "recommended_action": (
            "Monitor top categories separately and evaluate whether growth depends on a small number of segments."
        )
    }


def geographic_performance_insight(order_level):
    state_performance = (
        order_level
        .groupby("customer_state")
        .agg(
            avg_delay=("delivery_delay", "mean"),
            avg_review=("review_score", "mean"),
            orders=("order_id", "count")
        )
        .sort_values("orders", ascending=False)
    )

    top_state = state_performance.index[0]
    top_state_orders = state_performance.loc[top_state, "orders"]
    top_state_review = state_performance.loc[top_state, "avg_review"]

    return {
        "area": "Geographic Performance",
        "insight": (
            f"The state with the highest order volume is {top_state}, with "
            f"{top_state_orders:,} orders and an average review score of {top_state_review:.2f}."
        ),
        "business_implication": (
            "High-volume regions have a strong influence on overall marketplace performance."
        ),
        "recommended_action": (
            "Prioritize monitoring customer experience and logistics performance in high-volume states."
        )
    }


def freight_cost_insight(df):
    freight_summary = (
        df.groupby("product_category_name_english")
        .agg(
            avg_freight=("freight_value", "mean"),
            revenue=("revenue", "sum"),
            orders=("order_id", "nunique")
        )
        .sort_values("avg_freight", ascending=False)
    )

    top_freight_category = freight_summary.index[0]
    top_avg_freight = freight_summary.loc[top_freight_category, "avg_freight"]

    return {
        "area": "Freight Cost",
        "insight": (
            f"The category with the highest average freight cost is '{top_freight_category}', "
            f"with an average freight value of {top_avg_freight:.2f}."
        ),
        "business_implication": (
            "High freight costs may affect customer perception, conversion and profitability."
        ),
        "recommended_action": (
            "Analyze whether high freight categories also have lower review scores or lower repeat purchase potential."
        )
    }


def review_distribution_insight(order_level):
    review_distribution = (
        order_level["review_score"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    )

    five_star_share = review_distribution.loc[5]
    low_review_share = review_distribution.loc[[1, 2]].sum()

    return {
        "area": "Review Distribution",
        "insight": (
            f"Five-star reviews represent {five_star_share:.1f}% of reviewed orders, "
            f"while one and two-star reviews represent {low_review_share:.1f}%."
        ),
        "business_implication": (
            "The review distribution provides a high-level view of customer experience quality."
        ),
        "recommended_action": (
            "Investigate low-review orders to identify recurring operational, product or seller issues."
        )
    }


def generate_business_insights(df, order_level):
    insights = [
        delivery_performance_insight(order_level),
        satisfaction_insight(order_level),
        revenue_category_insight(df),
        geographic_performance_insight(order_level),
        freight_cost_insight(df),
        review_distribution_insight(order_level)
    ]

    return insights

### 7.3. Output Cell

In [117]:
insights = generate_business_insights(df, order_level)

for item in insights:
    print(f"\n{item['area']}")
    print("-" * len(item["area"]))
    print(f"Insight: {item['insight']}")
    print(f"Business implication: {item['business_implication']}")
    print(f"Recommended action: {item['recommended_action']}")


Delivery Performance
--------------------
Insight: Delivery performance relative to estimated dates is strong. Orders are delivered on average 11.9 days earlier than estimated, with an average delivery time of 12.1 days.
Business implication: The company appears to perform well against estimated delivery dates, possibly due to conservative estimates, efficient logistics, or both.
Recommended action: Continue monitoring delivery performance by region and seller to identify where early delivery contributes most to customer satisfaction.

Customer Satisfaction
---------------------
Insight: Five star orders are delivered 9.4 days earlier relative to expectations than one star orders.
Business implication: Customer satisfaction is associated with delivery performance relative to expectations, although other factors such as product quality, seller reliability and freight cost may also influence reviews.
Recommended action: Use delivery performance relative to estimated dates as a customer 

## 8. Automated Business Report

This section transforms structured analytical insights into a formatted business report.

The objective is to simulate how analysts communicate findings to stakeholders, moving from raw analysis to executive-ready outputs.

This step is critical in bridging the gap between data analysis and decision-making, ensuring that insights are clear, structured and actionable.

### 8.1. Generate Business Report

In [118]:
def generate_business_report(insights):
    report = []

    # Title
    report.append("# Automated Business Report")
    report.append("")

    # Executive Summary (IMPROVED)
    report.append("## Executive Summary")
    report.append(
        "This report highlights key drivers of customer satisfaction and operational performance in the marketplace. "
        "The analysis identifies delivery performance relative to expectations as a primary driver of customer experience, "
        "while revenue is concentrated in a limited number of product categories. "
        "Operational efficiency, particularly in logistics and freight cost management, plays a central role in both "
        "customer satisfaction and profitability."
    )
    report.append("")

    # Detailed Sections
    for item in insights:
        report.append(f"## {item['area']}")
        report.append("")

        report.append(f"**Insight:** {item['insight']}")
        report.append("")

        report.append(f"**Business implication:** {item['business_implication']}")
        report.append("")

        report.append(f"**Recommended action:** {item['recommended_action']}")
        report.append("")

    # 🔥 NEW: Conclusion (VERY IMPORTANT)
    report.append("## Conclusion")
    report.append(
        "Delivery performance relative to expectations emerges as a key driver of customer satisfaction. "
        "At the same time, revenue concentration and freight cost variability highlight areas of strategic and operational risk. "
        "Improving estimate accuracy and monitoring high-impact categories and regions can enhance both customer experience "
        "and profitability."
    )

    return "\n".join(report)

### 8.2. Run Report

In [119]:
business_report = generate_business_report(insights)

print(business_report)

# Automated Business Report

## Executive Summary
This report highlights key drivers of customer satisfaction and operational performance in the marketplace. The analysis identifies delivery performance relative to expectations as a primary driver of customer experience, while revenue is concentrated in a limited number of product categories. Operational efficiency, particularly in logistics and freight cost management, plays a central role in both customer satisfaction and profitability.

## Delivery Performance

**Insight:** Delivery performance relative to estimated dates is strong. Orders are delivered on average 11.9 days earlier than estimated, with an average delivery time of 12.1 days.

**Business implication:** The company appears to perform well against estimated delivery dates, possibly due to conservative estimates, efficient logistics, or both.

**Recommended action:** Continue monitoring delivery performance by region and seller to identify where early delivery contribute

### 8.3. Save as Markdown

In [120]:
import os

output_dir = "/content/drive/MyDrive/ai-ecommerce-insights-agent/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = output_dir + "/business_report.md"

with open(output_path, "w") as file:
    file.write(business_report)

print(f"Report saved to: {output_path}")

Report saved to: /content/drive/MyDrive/ai-ecommerce-insights-agent/outputs/business_report.md


## 9. Expanding the Insight Engine

The first version of the insight engine focused on delivery performance, customer satisfaction and revenue concentration.

This section expands the engine with additional business dimensions, making the system more useful as an AI assisted business analyst. The new areas include geographic performance, freight cost and review score distribution.

### 9.1. Geographic Performance Insight

In [121]:
def geographic_performance_insight(order_level):
    state_performance = (
        order_level
        .groupby("customer_state")
        .agg(
            avg_delay=("delivery_delay", "mean"),
            avg_review=("review_score", "mean"),
            orders=("order_id", "count")
        )
        .sort_values("orders", ascending=False)
    )

    top_state = state_performance.index[0]
    top_state_orders = state_performance.loc[top_state, "orders"]
    top_state_review = state_performance.loc[top_state, "avg_review"]

    return {
        "area": "Geographic Performance",
        "insight": (
            f"The state with the highest order volume is {top_state}, with "
            f"{top_state_orders:,} orders and an average review score of {top_state_review:.2f}."
        ),
        "business_implication": (
            "High-volume regions have a strong influence on overall marketplace performance."
        ),
        "recommended_action": (
            "Prioritize monitoring customer experience and logistics performance in high-volume states."
        )
    }

### 9.2. Freight Cost Insight

In [122]:
def freight_cost_insight(df):
    freight_summary = (
        df.groupby("product_category_name_english")
        .agg(
            avg_freight=("freight_value", "mean"),
            revenue=("revenue", "sum"),
            orders=("order_id", "nunique")
        )
        .sort_values("avg_freight", ascending=False)
    )

    top_freight_category = freight_summary.index[0]
    top_avg_freight = freight_summary.loc[top_freight_category, "avg_freight"]

    return {
        "area": "Freight Cost",
        "insight": (
            f"The category with the highest average freight cost is '{top_freight_category}', "
            f"with an average freight value of {top_avg_freight:.2f}."
        ),
        "business_implication": (
            "High freight costs may affect customer perception, conversion and profitability."
        ),
        "recommended_action": (
            "Analyze whether high freight categories also have lower review scores or lower repeat purchase potential."
        )
    }

### 9.3. Review Distribution Insight

In [123]:
def review_distribution_insight(order_level):
    review_distribution = (
        order_level["review_score"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    )

    five_star_share = review_distribution.loc[5]
    low_review_share = review_distribution.loc[[1, 2]].sum()

    return {
        "area": "Review Distribution",
        "insight": (
            f"Five-star reviews represent {five_star_share:.1f}% of reviewed orders, "
            f"while one and two-star reviews represent {low_review_share:.1f}%."
        ),
        "business_implication": (
            "The review distribution provides a high-level view of customer experience quality."
        ),
        "recommended_action": (
            "Investigate low-review orders to identify recurring operational, product or seller issues."
        )
    }

### 9.4. Update Main Agent Function

In [124]:
def generate_business_insights(df, order_level):
    insights = [
        delivery_performance_insight(order_level),
        satisfaction_insight(order_level),
        revenue_category_insight(df),
        geographic_performance_insight(order_level),
        freight_cost_insight(df),
        review_distribution_insight(order_level)
    ]

    return insights

### 9.5. Re-run Insights and Report

In [125]:
insights = generate_business_insights(df, order_level)

for item in insights:
    print(f"\n{item['area']}")
    print("-" * len(item["area"]))
    print(f"Insight: {item['insight']}")
    print(f"Business implication: {item['business_implication']}")
    print(f"Recommended action: {item['recommended_action']}")


Delivery Performance
--------------------
Insight: Delivery performance relative to estimated dates is strong. Orders are delivered on average 11.9 days earlier than estimated, with an average delivery time of 12.1 days.
Business implication: The company appears to perform well against estimated delivery dates, possibly due to conservative estimates, efficient logistics, or both.
Recommended action: Continue monitoring delivery performance by region and seller to identify where early delivery contributes most to customer satisfaction.

Customer Satisfaction
---------------------
Insight: Five star orders are delivered 9.4 days earlier relative to expectations than one star orders.
Business implication: Customer satisfaction is associated with delivery performance relative to expectations, although other factors such as product quality, seller reliability and freight cost may also influence reviews.
Recommended action: Use delivery performance relative to estimated dates as a customer 

In [126]:
business_report = generate_business_report(insights)

print(business_report)

# Automated Business Report

## Executive Summary
This report highlights key drivers of customer satisfaction and operational performance in the marketplace. The analysis identifies delivery performance relative to expectations as a primary driver of customer experience, while revenue is concentrated in a limited number of product categories. Operational efficiency, particularly in logistics and freight cost management, plays a central role in both customer satisfaction and profitability.

## Delivery Performance

**Insight:** Delivery performance relative to estimated dates is strong. Orders are delivered on average 11.9 days earlier than estimated, with an average delivery time of 12.1 days.

**Business implication:** The company appears to perform well against estimated delivery dates, possibly due to conservative estimates, efficient logistics, or both.

**Recommended action:** Continue monitoring delivery performance by region and seller to identify where early delivery contribute

In [127]:
output_path = output_dir + "/business_report.md"

with open(output_path, "w") as file:
    file.write(business_report)

print(f"Report saved to: {output_path}")

Report saved to: /content/drive/MyDrive/ai-ecommerce-insights-agent/outputs/business_report.md


## 10. Query Based Insight System

This section introduces a query routing mechanism that allows the system to respond dynamically to business questions.

Instead of generating only a static report, the system selects the appropriate analytical function based on user intent.

This represents a first step towards agentic AI, where the system decides which analytical tools to use depending on the question.

### 10.1. Function Definition

In [128]:
def answer_question(question, df, order_level):
    q = question.lower()

    intent_scores = {
        "delivery": sum(word in q for word in ["delivery", "logistics", "delay"]),
        "satisfaction": sum(word in q for word in ["satisfaction", "review", "customer experience"]),
        "revenue": sum(word in q for word in ["revenue", "sales", "category", "product"]),
        "geography": sum(word in q for word in ["state", "region", "geography", "location"]),
        "freight": sum(word in q for word in ["freight", "shipping cost", "delivery cost"]),
        "reviews": sum(word in q for word in ["distribution", "ratings", "reviews breakdown"])
    }

    # Select intents with score > 0, sorted by relevance
    intents = [k for k, v in sorted(intent_scores.items(), key=lambda x: x[1], reverse=True) if v > 0]

    responses = []

    for intent in intents:
        if intent == "delivery":
            responses.append(delivery_performance_insight(order_level))
        elif intent == "satisfaction":
            responses.append(satisfaction_insight(order_level))
        elif intent == "revenue":
            responses.append(revenue_category_insight(df))
        elif intent == "geography":
            responses.append(geographic_performance_insight(order_level))
        elif intent == "freight":
            responses.append(freight_cost_insight(df))
        elif intent == "reviews":
            responses.append(review_distribution_insight(order_level))

    if not responses:
        return "I could not determine the appropriate analysis for this question."

    formatted = []

    for result in responses:
        formatted.append(
            f"\n{result['area']}\n"
            + "-" * len(result["area"]) + "\n"
            f"Insight: {result['insight']}\n"
            f"Business implication: {result['business_implication']}\n"
            f"Recommended action: {result['recommended_action']}\n"
        )

    return "\n".join(formatted)

### 10.2. Test the Query System

In [129]:
print(answer_question("How is delivery performance?", df, order_level))
print()
print(answer_question("What drives customer satisfaction?", df, order_level))
print()
print(answer_question("Which categories generate the most revenue?", df, order_level))
print()
print(answer_question("Tell me about freight costs and customer reviews", df, order_level))


Delivery Performance
--------------------
Insight: Delivery performance relative to estimated dates is strong. Orders are delivered on average 11.9 days earlier than estimated, with an average delivery time of 12.1 days.
Business implication: The company appears to perform well against estimated delivery dates, possibly due to conservative estimates, efficient logistics, or both.
Recommended action: Continue monitoring delivery performance by region and seller to identify where early delivery contributes most to customer satisfaction.



Customer Satisfaction
---------------------
Insight: Five star orders are delivered 9.4 days earlier relative to expectations than one star orders.
Business implication: Customer satisfaction is associated with delivery performance relative to expectations, although other factors such as product quality, seller reliability and freight cost may also influence reviews.
Recommended action: Use delivery performance relative to estimated dates as a custome

## 11. Advanced Query Reasoning Layer

The initial query system uses simple keyword matching to route questions to analytical functions.

This section improves the reasoning capability of the agent by introducing:
- intent classification
- multi-topic questions
- more flexible interpretation of user queries

This moves the system closer to a true agentic architecture, where the system understands the question rather than relying only on keyword matching.

### 11.1. Intent Classification Layer

In [130]:
def classify_intent(question):
    q = question.lower()

    intent_map = {
        "delivery": ["delivery", "logistics", "delay", "shipping speed"],
        "satisfaction": ["satisfaction", "review", "rating", "customer experience"],
        "revenue": ["revenue", "sales", "category", "product", "income"],
        "geography": ["state", "region", "location", "geography"],
        "freight": ["freight", "shipping cost", "delivery cost"],
        "reviews_dist": ["distribution", "ratings breakdown", "review distribution"]
    }

    intent_scores = {
        intent: sum(word in q for word in keywords)
        for intent, keywords in intent_map.items()
    }

    intents = [
        k for k, v in sorted(intent_scores.items(), key=lambda x: x[1], reverse=True)
        if v > 0
    ]

    return intents

### 11.2. Tool Mapping Layer

In [131]:
def run_intent(intent, df, order_level):
    if intent == "delivery":
        return delivery_performance_insight(order_level)

    elif intent == "satisfaction":
        return satisfaction_insight(order_level)

    elif intent == "revenue":
        return revenue_category_insight(df)

    elif intent == "geography":
        return geographic_performance_insight(order_level)

    elif intent == "freight":
        return freight_cost_insight(df)

    elif intent == "reviews_dist":
        return review_distribution_insight(order_level)

    else:
        return {
            "area": "Unknown",
            "insight": "No insight available for this query.",
            "business_implication": "",
            "recommended_action": ""
        }

In [132]:
def rank_insights(insights):
    priority_order = {
        "Customer Satisfaction": 3,
        "Delivery Performance": 3,
        "Revenue Concentration": 2,
        "Freight Cost": 2,
        "Geographic Performance": 1,
        "Review Distribution": 1
    }

    return sorted(
        insights,
        key=lambda x: priority_order.get(x["area"], 0),
        reverse=True
    )

### 11.3. Improved Answer System

In [133]:
def advanced_answer_question(question, df, order_level):
    intents = classify_intent(question)

    if not intents:
        return "I could not understand the question. Try asking about delivery, revenue, satisfaction or geography."

    results = [run_intent(intent, df, order_level) for intent in intents]
    results = rank_insights(results)

    response = []

    response.append("# Question")
    response.append(question)
    response.append("")

    response.append("## Executive Answer")

    if "delivery" in intents and "satisfaction" in intents:
        delivery = delivery_performance_insight(order_level)
        satisfaction = satisfaction_insight(order_level)

        response.append(
            f"Delivery performance is a key driver of customer satisfaction. "
            f"{delivery['insight']} {satisfaction['insight']}"
        )

    elif "revenue" in intents:
        revenue = revenue_category_insight(df)

        response.append(
            f"Revenue is concentrated in key categories. {revenue['insight']}"
        )

    else:
        response.append(
            "The analysis highlights key business drivers across the requested areas, "
            "including performance metrics and operational implications."
        )

    response.append("")
    response.append("## Key Takeaways")

    for r in results[:3]:
        response.append(f"- {r['insight']}")

    response.append("")
    response.append("## Detailed Insights")
    response.append("")

    for result in results:
        response.append(f"### {result['area']}")
        response.append("")
        response.append(f"**Insight:** {result['insight']}")
        response.append("")
        response.append(f"**Business implication:** {result['business_implication']}")
        response.append("")
        response.append(f"**Recommended action:** {result['recommended_action']}")
        response.append("")

    return "\n".join(response)

### 11.4. Test

In [134]:
from IPython.display import Markdown, display

display(Markdown(advanced_answer_question(
    "What drives customer satisfaction and how is delivery performing?",
    df,
    order_level
)))

display(Markdown(advanced_answer_question(
    "Which categories generate the most revenue and what about freight costs?",
    df,
    order_level
)))

display(Markdown(advanced_answer_question(
    "What are the biggest risks related to revenue concentration, freight cost and customer satisfaction?",
    df,
    order_level
)))

# Question
What drives customer satisfaction and how is delivery performing?

## Executive Answer
Delivery performance is a key driver of customer satisfaction. Delivery performance relative to estimated dates is strong. Orders are delivered on average 11.9 days earlier than estimated, with an average delivery time of 12.1 days. Five star orders are delivered 9.4 days earlier relative to expectations than one star orders.

## Key Takeaways
- Delivery performance relative to estimated dates is strong. Orders are delivered on average 11.9 days earlier than estimated, with an average delivery time of 12.1 days.
- Five star orders are delivered 9.4 days earlier relative to expectations than one star orders.

## Detailed Insights

### Delivery Performance

**Insight:** Delivery performance relative to estimated dates is strong. Orders are delivered on average 11.9 days earlier than estimated, with an average delivery time of 12.1 days.

**Business implication:** The company appears to perform well against estimated delivery dates, possibly due to conservative estimates, efficient logistics, or both.

**Recommended action:** Continue monitoring delivery performance by region and seller to identify where early delivery contributes most to customer satisfaction.

### Customer Satisfaction

**Insight:** Five star orders are delivered 9.4 days earlier relative to expectations than one star orders.

**Business implication:** Customer satisfaction is associated with delivery performance relative to expectations, although other factors such as product quality, seller reliability and freight cost may also influence reviews.

**Recommended action:** Use delivery performance relative to estimated dates as a customer experience KPI and expand the analysis to include product category, seller performance and freight value.


# Question
Which categories generate the most revenue and what about freight costs?

## Executive Answer
Revenue is concentrated in key categories. The top revenue generating category is 'health_beauty', generating 1,491,397.76 and representing 9.1% of analyzed revenue.

## Key Takeaways
- The top revenue generating category is 'health_beauty', generating 1,491,397.76 and representing 9.1% of analyzed revenue.
- The category with the highest average freight cost is 'computers', with an average freight value of 48.01.

## Detailed Insights

### Revenue Concentration

**Insight:** The top revenue generating category is 'health_beauty', generating 1,491,397.76 and representing 9.1% of analyzed revenue.

**Business implication:** Revenue is concentrated in key categories, which may represent both growth opportunities and dependency risks.

**Recommended action:** Monitor top categories separately and evaluate whether growth depends on a small number of segments.

### Freight Cost

**Insight:** The category with the highest average freight cost is 'computers', with an average freight value of 48.01.

**Business implication:** High freight costs may affect customer perception, conversion and profitability.

**Recommended action:** Analyze whether high freight categories also have lower review scores or lower repeat purchase potential.


# Question
What are the biggest risks related to revenue concentration, freight cost and customer satisfaction?

## Executive Answer
Revenue is concentrated in key categories. The top revenue generating category is 'health_beauty', generating 1,491,397.76 and representing 9.1% of analyzed revenue.

## Key Takeaways
- Five star orders are delivered 9.4 days earlier relative to expectations than one star orders.
- The top revenue generating category is 'health_beauty', generating 1,491,397.76 and representing 9.1% of analyzed revenue.
- The category with the highest average freight cost is 'computers', with an average freight value of 48.01.

## Detailed Insights

### Customer Satisfaction

**Insight:** Five star orders are delivered 9.4 days earlier relative to expectations than one star orders.

**Business implication:** Customer satisfaction is associated with delivery performance relative to expectations, although other factors such as product quality, seller reliability and freight cost may also influence reviews.

**Recommended action:** Use delivery performance relative to estimated dates as a customer experience KPI and expand the analysis to include product category, seller performance and freight value.

### Revenue Concentration

**Insight:** The top revenue generating category is 'health_beauty', generating 1,491,397.76 and representing 9.1% of analyzed revenue.

**Business implication:** Revenue is concentrated in key categories, which may represent both growth opportunities and dependency risks.

**Recommended action:** Monitor top categories separately and evaluate whether growth depends on a small number of segments.

### Freight Cost

**Insight:** The category with the highest average freight cost is 'computers', with an average freight value of 48.01.

**Business implication:** High freight costs may affect customer perception, conversion and profitability.

**Recommended action:** Analyze whether high freight categories also have lower review scores or lower repeat purchase potential.


## 12. Saving Query Based Outputs

This section saves example query responses generated by the agent.

Saving outputs makes the project easier to document, review and include in a GitHub portfolio.

### 12.1. Building Queries

In [135]:
query_1 = advanced_answer_question(
    "What drives customer satisfaction and how is delivery performing?",
    df,
    order_level
)

query_2 = advanced_answer_question(
    "Which categories generate the most revenue and what about freight costs?",
    df,
    order_level
)

query_3 = advanced_answer_question(
    "What are the biggest risks related to revenue concentration, freight cost and customer satisfaction?",
    df,
    order_level
)

### 12.2. Query Output

In [136]:
import os
import re

query_output_dir = "/content/drive/MyDrive/ai-ecommerce-insights-agent/outputs/query_responses"
os.makedirs(query_output_dir, exist_ok=True)

def save_query_output(name, content):
    safe_name = re.sub(r'[^a-zA-Z0-9_]', '_', name.lower())[:50]
    path = f"{query_output_dir}/{safe_name}.md"

    with open(path, "w") as file:
        file.write(content)

    return path

path1 = save_query_output("customer_satisfaction_and_delivery", query_1)
path2 = save_query_output("revenue_and_freight", query_2)
path3 = save_query_output("business_risks", query_3)

print("Saved to:")
print(path1)
print(path2)
print(path3)

Saved to:
/content/drive/MyDrive/ai-ecommerce-insights-agent/outputs/query_responses/customer_satisfaction_and_delivery.md
/content/drive/MyDrive/ai-ecommerce-insights-agent/outputs/query_responses/revenue_and_freight.md
/content/drive/MyDrive/ai-ecommerce-insights-agent/outputs/query_responses/business_risks.md


## 13. Free Local LLM Rewrite Layer

This section adds a free local language model layer to improve the wording of the agent's responses.

The analytical logic remains rule based and transparent. The language model is only used to rewrite the final response in a clearer business communication style.

This avoids paid APIs while still demonstrating how LLMs can enhance an AI-assisted analytics workflow.

### 13.1 Install and Import Libraries

In [137]:
!pip install transformers sentencepiece -q

In [138]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

### 13.2 Load FLAN-T5

In [139]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


### 13.3. Create Rewrite Function

In [140]:
def rewrite_with_local_llm(text):
    prompt = f"""
You are a senior business analyst.

Rewrite the following report to improve clarity, professionalism and flow.

STRICT RULES:
- DO NOT remove any information
- DO NOT remove numbers
- DO NOT summarize
- KEEP ALL HEADINGS EXACTLY as they are
- KEEP bullet points and structure
- ONLY improve wording and readability

TEXT:
{text}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    outputs = model.generate(
    **inputs,
    max_new_tokens=800,
    do_sample=False
)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

### 13.4. Test with One Query Response

In [141]:
from IPython.display import Markdown, display

raw_response = advanced_answer_question(
    "What drives customer satisfaction and how is delivery performing?",
    df,
    order_level
)

polished_response = rewrite_with_local_llm(raw_response)

display(Markdown(polished_response))

### Executive Answer Delivery performance is a key driver of customer satisfaction. Delivery performance relative to estimated dates is strong. Orders are delivered on average 11.9 days earlier than estimated, with an average delivery time of 12.1 days. Five star orders are delivered 9.4 days earlier relative to expectations than one star orders. ## Key Takeaways - Delivery performance relative to estimated dates is strong. Orders are delivered on average 11.9 days earlier than estimated, with an average delivery time of 12.1 days. - Five star orders are delivered 9.4 days earlier relative to expectations than one star orders. ## Detailed Insights ### Delivery Performance **Insight:** Five star orders are delivered 9.4 days earlier relative to expectations than one star orders. **Recommended action:** Use delivery performance relative to estimated dates as a customer experience KPI and expand the analysis to include product category, seller performance and freight value.

## 14. Design Decisions

This project was designed with a hybrid architecture:

- A deterministic analytics layer ensures accuracy and transparency
- A rule-based insight engine generates consistent business insights
- A query routing system enables dynamic interaction with the data
- An optional LLM layer was implemented to enhance language quality
- An intent-based query routing layer with prioritization logic to simulate agent-like reasoning

Due to limitations of lightweight local models, the LLM is kept optional to ensure reliability and avoid unnecessary complexity or cost.

This approach reflects real-world production systems, where deterministic logic is preferred for critical business decisions, and LLMs are used selectively.

## 15. Final Conclusion

This project demonstrates how data analytics and AI can be combined to generate structured, business-oriented insights in an e-commerce environment.

### Key Findings

- Delivery performance exceeds expectations, with orders arriving significantly earlier than estimated on average.
- Delivery performance is a leading indicator of customer satisfaction and should be treated as a core business KPI.
- Customer satisfaction is strongly associated with delivery performance relative to expectations, rather than absolute delivery time.
- Revenue is concentrated in a limited number of product categories, indicating both growth opportunities and dependency risks.
- Freight costs vary significantly across categories and may impact customer perception and conversion.

### Business Recommendations

1. **Use delivery performance as a strategic lever**
   - Track delivery vs estimated time as a core KPI
   - Optimize logistics in high-impact regions and sellers

2. **Improve customer experience drivers**
   - Combine delivery metrics with product quality and seller performance
   - Investigate low-review segments systematically

3. **Manage revenue concentration risk**
   - Diversify category growth strategy
   - Invest in underperforming but high-potential categories

4. **Optimize freight cost structure**
   - Analyze high-cost categories for margin and satisfaction impact
   - Explore pricing, bundling or logistics optimization strategies

### Final Takeaway

The analysis highlights that operational efficiency, particularly in logistics, plays a critical role in customer satisfaction and overall marketplace performance.

By integrating analytics with structured insight generation and AI-assisted communication, this project simulates how modern data teams can move beyond reporting into decision support.

This project goes beyond traditional dashboards by introducing an intent-driven analytics agent capable of dynamically selecting and prioritizing business insights. This reflects a shift toward AI-assisted decision systems, where analytical reasoning and communication are integrated into a single workflow.

## 16. Limitations and Next Steps

While this project demonstrates a structured approach to analytics and AI-assisted insight generation, several limitations should be considered:

- The insight engine is rule-based and relies on predefined logic rather than fully autonomous reasoning
- The query system is keyword and intent-based, which limits flexibility for complex or ambiguous questions
- The local LLM layer is lightweight and may not preserve full structure in longer outputs

## Next Steps

Future improvements could include:

- integrating a more advanced LLM for improved reasoning and language quality
- expanding the agent to include additional data sources (e.g. seller performance, product-level features)
- building an interactive interface (e.g. Streamlit) for real-time querying
- introducing feedback loops to refine insights based on user interaction